# Retail Customer & Transaction Data Cleaning

Cleaning and joining raw customer and transaction tables with pandas: parsing currency strings, standardizing categories, imputing missing ages, and merging to flag transactions without a customer profile.

_Datasets are not included in this repository._

In [65]:
import pandas as pd
df = pd.read_csv('/content/customers_raw.csv')
df.head()
print(df.head(100))

   CustomerID    JoinDate   Age  Gender     City
0    CUST_001  01-01-2024  30.0       M    Medan
1    CUST_002  02-01-2024  50.0  Female    Medan
2    CUST_003  03-01-2024  40.0    Male  Bandung
3    CUST_004  04-01-2024  35.0       F  Jakarta
4    CUST_005  05-01-2024  25.0     NaN    Medan
..        ...         ...   ...     ...      ...
95   CUST_096  05-04-2024  30.0    Male  Bandung
96   CUST_097  06-04-2024  35.0  Female  Bandung
97   CUST_098  07-04-2024  30.0  female    Medan
98   CUST_099  08-04-2024  20.0  female  Jakarta
99   CUST_100  09-04-2024  20.0  Female    Medan

[100 rows x 5 columns]


In [66]:
import math as math
df2 = pd.read_csv('/content/transactions_raw.csv')
df2.Amount = df2.Amount.str.replace('IDR ', '')
df2.Amount = df2.Amount.str.replace(',', '')
df2.Amount = df2.Amount.astype(float)
df2.Amount = df2.Amount.apply(lambda x: abs(x))
df2.head()

,TransactionID,Cust_ID,TransactionDate,Amount,Category,IsReturned
0,TX_0001,CUST_063,2025-01-01 00:00:00,300000.0,Groceries,0
1,TX_0002,CUST_085,2025-01-01 01:00:00,150000.0,Fashion,0
2,TX_0003,CUST_032,2025-01-01 02:00:00,150000.0,Groceries,0
3,TX_0004,CUST_087,2025-01-01 03:00:00,1200000.0,Electronics,0
4,TX_0005,CUST_033,2025-01-01 04:00:00,450000.0,Fashion,1


In [67]:
def gendering(x):
  x = str(x).upper()
  if x == "M" or x == "MALE":
    return "Male"
  elif x == "F" or x == "FEMALE":
    return "Female"
  else:
    return "Unknown"
df.Gender = df.Gender.apply(gendering)
print(df.head(100))

   CustomerID    JoinDate   Age   Gender     City
0    CUST_001  01-01-2024  30.0     Male    Medan
1    CUST_002  02-01-2024  50.0   Female    Medan
2    CUST_003  03-01-2024  40.0     Male  Bandung
3    CUST_004  04-01-2024  35.0   Female  Jakarta
4    CUST_005  05-01-2024  25.0  Unknown    Medan
..        ...         ...   ...      ...      ...
95   CUST_096  05-04-2024  30.0     Male  Bandung
96   CUST_097  06-04-2024  35.0   Female  Bandung
97   CUST_098  07-04-2024  30.0   Female    Medan
98   CUST_099  08-04-2024  20.0   Female  Jakarta
99   CUST_100  09-04-2024  20.0   Female    Medan

[100 rows x 5 columns]


In [68]:
# Ukuran data akan berkurang yang akan menyebabkan bias yang lebih tinggi karena semakin tidak terbalance akibar proporsi yang tidak rata
yyy =df.groupby("City")["Age"].median()
df.Age = df.Age.fillna(df.City.map(yyy))
df

,CustomerID,JoinDate,Age,Gender,City
0,CUST_001,01-01-2024,30.0,Male,Medan
1,CUST_002,02-01-2024,50.0,Female,Medan
2,CUST_003,03-01-2024,40.0,Male,Bandung
3,CUST_004,04-01-2024,35.0,Female,Jakarta
4,CUST_005,05-01-2024,25.0,Unknown,Medan
...,...,...,...,...,...
95,CUST_096,05-04-2024,30.0,Male,Bandung
96,CUST_097,06-04-2024,35.0,Female,Bandung
97,CUST_098,07-04-2024,30.0,Female,Medan
98,CUST_099,08-04-2024,20.0,Female,Jakarta


In [69]:
merge = pd.merge(df2, df, left_on="Cust_ID", right_on="CustomerID", how = "left",indicator=True)
merge["TanpaProfil"]= merge.CustomerID.isna()
merge
##merge.TanpaProfil.value_counts()

,TransactionID,Cust_ID,TransactionDate,Amount,Category,IsReturned,CustomerID,JoinDate,Age,Gender,City,_merge,TanpaProfil
0,TX_0001,CUST_063,2025-01-01 00:00:00,300000.0,Groceries,0,CUST_063,03-03-2024,30.0,Female,Surabaya,both,False
1,TX_0002,CUST_085,2025-01-01 01:00:00,150000.0,Fashion,0,CUST_085,25-03-2024,30.0,Male,Jakarta,both,False
2,TX_0003,CUST_032,2025-01-01 02:00:00,150000.0,Groceries,0,CUST_032,01-02-2024,25.0,Male,Surabaya,both,False
3,TX_0004,CUST_087,2025-01-01 03:00:00,1200000.0,Electronics,0,CUST_087,27-03-2024,40.0,Female,Bandung,both,False
4,TX_0005,CUST_033,2025-01-01 04:00:00,450000.0,Fashion,1,CUST_033,02-02-2024,20.0,Male,Jakarta,both,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,TX_0296,CUST_040,2025-01-13 07:00:00,450000.0,Fashion,1,CUST_040,09-02-2024,30.0,Unknown,Bandung,both,False
296,TX_0297,CUST_042,2025-01-13 08:00:00,450000.0,Fashion,0,CUST_042,11-02-2024,30.0,Male,Medan,both,False
297,TX_0298,CUST_009,2025-01-13 09:00:00,300000.0,Groceries,0,CUST_009,09-01-2024,35.0,Male,Medan,both,False
298,TX_0299,CUST_050,2025-01-13 10:00:00,450000.0,Fashion,0,CUST_050,19-02-2024,25.0,Male,Surabaya,both,False


In [71]:
loyal = merge[merge.IsReturned ==0]
loyal

,TransactionID,Cust_ID,TransactionDate,Amount,Category,IsReturned,CustomerID,JoinDate,Age,Gender,City,_merge,TanpaProfil
0,TX_0001,CUST_063,2025-01-01 00:00:00,300000.0,Groceries,0,CUST_063,03-03-2024,30.0,Female,Surabaya,both,False
1,TX_0002,CUST_085,2025-01-01 01:00:00,150000.0,Fashion,0,CUST_085,25-03-2024,30.0,Male,Jakarta,both,False
2,TX_0003,CUST_032,2025-01-01 02:00:00,150000.0,Groceries,0,CUST_032,01-02-2024,25.0,Male,Surabaya,both,False
3,TX_0004,CUST_087,2025-01-01 03:00:00,1200000.0,Electronics,0,CUST_087,27-03-2024,40.0,Female,Bandung,both,False
5,TX_0006,CUST_067,2025-01-01 05:00:00,450000.0,Electronics,0,CUST_067,07-03-2024,20.0,Male,Medan,both,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,TX_0295,CUST_013,2025-01-13 06:00:00,50000.0,Electronics,0,CUST_013,13-01-2024,30.0,Unknown,Surabaya,both,False
296,TX_0297,CUST_042,2025-01-13 08:00:00,450000.0,Fashion,0,CUST_042,11-02-2024,30.0,Male,Medan,both,False
297,TX_0298,CUST_009,2025-01-13 09:00:00,300000.0,Groceries,0,CUST_009,09-01-2024,35.0,Male,Medan,both,False
298,TX_0299,CUST_050,2025-01-13 10:00:00,450000.0,Fashion,0,CUST_050,19-02-2024,25.0,Male,Surabaya,both,False
